In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from pathlib import Path

RAW_PATH = Path("../data/raw")
PROCESSED_PATH = Path("../data/processed")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

movies = pd.read_csv(RAW_PATH/ "movies_metadata.csv")
keywords = pd.read_csv(RAW_PATH/ "keywords.csv")
credits = pd.read_csv(RAW_PATH/ "credits.csv")

C:\Users\Patrycja\AppData\Local\Temp\ipykernel_11196\3808944297.py:8: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv(RAW_PATH/ "movies_metadata.csv")


In [3]:
columns = ["id", "title", "overview", "genres", "runtime", "vote_average", "vote_count", "release_date", "original_language", "popularity", "imdb_id"]

df = movies[columns].copy()

## Konwersja typu danych 

In [4]:
df["id"] = pd.to_numeric(df["id"], errors="coerce")
numeric_cols = ["runtime", "vote_average", "vote_count", "popularity"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

df = df.dropna(subset=["id", "title"])

df["id"] = df["id"].astype("int64")
df["release_year"] = df["release_date"].dt.year

## Usuwanie duplikatów

In [5]:
print("Duplicated IDs: ", df["id"].duplicated().sum())
df = df.drop_duplicates(subset=["id"], keep="first").copy()

Duplicated IDs:  30


## Przetwarzanie gatunków

In [6]:
import ast

def extract_names(value):
    if pd.isna(value):
        return []

    try:
        data = ast.literal_eval(value)

        if not isinstance(data, list):
            return []

        return [
            item["name"] for item in data if isinstance(item, dict) and item.get("name")
        ]

    except (ValueError, SyntaxError, TypeError):
        return []

In [7]:
df["genres_list"] = df["genres"].apply(extract_names)
display(df[["title", "genres_list"]].head())

,title,genres_list
0,Toy Story,"[Animation, Comedy, Family]"
1,Jumanji,"[Adventure, Fantasy, Family]"
2,Grumpier Old Men,"[Romance, Comedy]"
3,Waiting to Exhale,"[Comedy, Drama, Romance]"
4,Father of the Bride Part II,[Comedy]


## Keywords

In [8]:
keywords["id"] = pd.to_numeric(keywords["id"], errors="coerce")

keywords = keywords.dropna(subset=["id"]).copy()
keywords["id"] = keywords["id"].astype("int64")

keywords["keywords_list"] = keywords["keywords"].apply(extract_names)

In [9]:
keywords_clean = keywords.groupby("id")["keywords_list"].agg(
    lambda lists: list(dict.fromkeys(
        name for items in lists for name in items
    ))
).reset_index()

df = df.merge(
    keywords_clean,
    on="id",
    how="left",
    validate="one_to_one"
)

df["keywords_list"] = df["keywords_list"].apply(
    lambda x: x if isinstance(x, list) else []
)

## Obsada i reżyserzy

In [10]:
credits["id"] = pd.to_numeric(credits["id"], errors="coerce")
credits = credits.dropna(subset=["id"]).copy()
credits["id"] = credits["id"].astype("int64")

In [11]:
def extract_directors(value):
    if pd.isna(value):
        return []

    try:
        crew = ast.literal_eval(value)

        return list(dict.fromkeys(
            person["name"]
            for person in crew
            if person.get("job") == "Director"
            and person.get("name")
        ))

    except (ValueError, SyntaxError, TypeError):
        return []


def extract_cast(value):
    if pd.isna(value):
        return []

    try:
        cast = ast.literal_eval(value)

        return [
            person["name"]
            for person in cast[:5]
            if person.get("name")
        ]

    except (ValueError, SyntaxError, TypeError):
        return []

In [12]:
credits["directors"] = credits["crew"].apply(extract_directors)
credits["top_cast"] = credits["cast"].apply(extract_cast)

credits_clean = credits[["id", "directors", "top_cast"]].drop_duplicates(subset=["id"])

df = df.merge(
    credits_clean,
    on="id",
    how="left",
    validate="one_to_one"
)

for col in ["directors", "top_cast"]:
    df[col] = df[col].apply(
        lambda x: x if isinstance(x, list) else []
    )

## Czyszczenie opisów fabuł

In [13]:
df["overview"] = (
    df["overview"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [14]:
print("Movies without overview:", (df["overview"] == "").sum())

Movies without overview: 959


## Przygotowanie tekstu pod Sentence-Transformer

In [15]:
df["keywords_text"] = df["keywords_list"].apply(
    lambda x: ", ".join(x)
)

df["genres_text"] = df["genres_list"].apply(
    lambda x: ", ".join(x)
)

In [16]:
def prepare_embedding_text(row):
    parts = []

    if row["overview"]:
        parts.append(
            f"Plot: {row["overview"]}"
        )

    if row["keywords_text"]:
        parts.append(
            f"Keywords: {row["keywords_text"]}"
        )

    return " ".join(parts)

df["embedding_text"] = df.apply(prepare_embedding_text, axis=1)

## Przygotowanie cech do rankingu

In [17]:
df.loc[df["runtime"] <= 0, "runtime"] = np.nan
df.loc[~df["vote_average"].between(0,10), "vote_average"] = np.nan
df.loc[df["vote_count"] < 0, "vote_count"] = np.nan

In [18]:
df["runtime_category"] = pd.cut(
    df["runtime"],
    bins=[0, 90, 120, 150, np.inf],
    labels = ["short", "medium", "long", "very_long"]
)

## Kontrola

In [19]:
print("Dataset shape: ", df.shape)
print("Unique movies: ", df["id"].nunique())

print("Missing overviews: ", (df["overview"]=="").sum())
print("Missing genres: ", df["genres_list"].apply(len).eq(0).sum())
print("Missing keywords: ", df["keywords_list"].apply(len).eq(0).sum())

print("Movies with embedding text: ", df["embedding_text"].str.len().gt(0).sum())

assert df["id"].is_unique

Dataset shape:  (45430, 20)
Unique movies:  45430
Missing overviews:  959
Missing genres:  2442
Missing keywords:  14341
Movies with embedding text:  44676


In [20]:
display(df[
    ["id", "title", "overview", "genres_list", "keywords_list", "runtime", "vote_average", "embedding_text"]
].head())

,id,title,overview,genres_list,keywords_list,runtime,vote_average,embedding_text
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[Animation, Comedy, Family]","[jealousy, toy, boy, friendship, friends, riva...",81.0,7.7,"Plot: Led by Woody, Andy's toys live happily i..."
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,"[Adventure, Fantasy, Family]","[board game, disappearance, based on children'...",104.0,6.9,Plot: When siblings Judy and Peter discover an...
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[Romance, Comedy]","[fishing, best friend, duringcreditsstinger, o...",101.0,6.5,Plot: A family wedding reignites the ancient f...
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[Comedy, Drama, Romance]","[based on novel, interracial relationship, sin...",127.0,6.1,"Plot: Cheated on, mistreated and stepped on, t..."
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,[Comedy],"[baby, midlife crisis, confidence, aging, daug...",106.0,5.7,Plot: Just when George Banks has recovered fro...


In [21]:
df.to_parquet(PROCESSED_PATH / "movies_processed.parquet", index=False)